# Comparative Analysis of Gold vs. Global Equities (2000–2026)
## A Cross-Regional Study on Inflation Hedging, Asset Efficiency, and Currency Impact

---

### **Executive Summary**
* **Time Horizon:** 26 Years (2000–2026) with Sensitivity Analysis at 2021.
* **Regions:** USA (USD), India (INR), and Germany (EUR).
* **Objective:** Evaluating Gold’s regional performance and against S&P 500, Nifty 50, and DAX 40 through the lens of localized inflation and currency volatility.

---

### **Notebook Structure**
1. **Data Ingestion & Preprocessing:** Localization and Inflation Adjustment.
2. **The Ferri Test:** Annualized Real Gains vs. CPI.
3. **Correlation Dynamics:** Portfolio Diversification Mapping.
4. **Efficiency Metrics:** Sharpe Ratio & Volatility Benchmarking.
5. **Temporal Consistency:** Rolling 3-Year Risk-Adjusted Returns.
6. **Market Liquidity:** Volume Trends.
7. **Return Attribution:** Gold Price vs. Currency Devaluation vs. Compounding.
8. **Wealth Simulation:** Regional DCA Performance under Realistic Friction & Taxes.

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load the variables from .env
load_dotenv()

# Access the hidden path
data_dir = os.getenv('DATA_DIR')

charts_dir = os.path.join(data_dir, 'charts')
if not os.path.exists(charts_dir):
    os.makedirs(charts_dir)

## 1. Data Ingestion & Preprocessing (2000–2026)

This section prepares the core dataset for our sensitivity analysis by localizing prices, adjusting for inflation, and calculating returns for the period ending **April 15, 2026**.

### Key Operations:
* **Sensitivity Analysis :** Two analysis must be performed since after looking at the sudden surge in volume after the end of 2021. For case 1 run the entire dataset, for case 2 Filter the dataset to end at **2021**. This allows us to compare results against the full 2026 dataset to see how post-2022 volume surges and geopolitical shifts impacted asset efficiency.
* **Returns Calculation:** Generates monthly percentage changes for correlation and risk-adjusted metric analysis.

In [ ]:
# 1. Load the master dataset
df = pd.read_csv(os.path.join(data_dir, 'ULTIMATE_financial_master_WITH_CPI.csv'), index_col=0, parse_dates=True)

# 2. Localize Gold Prices (Currency Conversion)
df.rename(columns={'Gold_Close': 'Gold_USD'}, inplace=True)
df['Gold_INR'] = df['Gold_USD'] * df['USD_INR']
df['Gold_EUR'] = df['Gold_USD'] * df['USD_EUR']

# 3. Calculate Real (Inflation-Adjusted) Prices
# Formula: Nominal_Price * (CPI_at_Base / CPI_at_Current)
# Use the most recent CPI value as the "Base" to see prices in '2026 money'
usa_base_cpi = df['CPI_USA'].iloc[-1]
india_base_cpi = df['CPI_India'].iloc[-1]
germany_base_cpi = df['CPI_Germany'].iloc[-1]

df['Real_Gold_USD'] = df['Gold_USD'] * (usa_base_cpi / df['CPI_USA'])
df['Real_Gold_INR'] = df['Gold_INR'] * (india_base_cpi / df['CPI_India'])
df['Real_Gold_EUR'] = df['Gold_EUR'] * (germany_base_cpi / df['CPI_Germany'])

# 4. Calculate Monthly Returns (for Correlation Analysis)
# Resample to 'ME' (Month End) to align with CPI frequency
# 1. Define the specific columns for the correlation study
# Exclude Volume and raw exchange rates to focus on Prices, Stocks, and CPI
analysis_cols = [
    'Gold_USD', 'Gold_INR', 'Gold_EUR', 
    'SP500', 'DAX', 'Nifty50', 
    'CPI_USA', 'CPI_India', 'CPI_Germany',
    'Real_Gold_USD', 'Real_Gold_INR', 'Real_Gold_EUR'
]

# comment this code out for sensitivity analysis (2000 - 2021)
df_monthly = df.resample('ME').last()
df_returns = df_monthly[analysis_cols].pct_change()
# Drop the first row (which will be NaN after pct_change)
df_returns = df_returns.dropna()

# run this code for the sensitivity analysis (2000 - 2021)
#df_monthly_full = df.resample('ME').last()
# 2. Slice the data to end at 2021
# This creates your "Pre-Crisis / Pre-Volume Surge" dataset
#df_monthly = df_monthly_full[:'2021-12-31']

# 3. Calculate returns on the filtered data
#df_returns = df_monthly[analysis_cols].pct_change().dropna()

print("Returns calculated for analysis columns")

## 2. To test Richard A Ferri's up to date hypotheis: Gold CAGR vs. Inflation

This section evaluates Gold's performance as an inflation hedge by comparing its **Compound Annual Growth Rate (CAGR)** against the national **Inflation Rate** for each region.

### Key Details:
* **CAGR Logic:** Standardizes multi-year growth into a single annual price series.
* **Real Gain Calculation:** Subtracts the Inflation CAGR from the Gold CAGR. A positive result indicates that Gold successfully preserved and increased purchasing clarity.

In [ ]:
def calculate_cagr(series):
    """Calculates Compound Annual Growth Rate (CAGR) for a given price series"""
    start_val = series.iloc[0]
    end_val = series.iloc[-1]
    n_years = len(series) / 12
    return (end_val / start_val) ** (1 / n_years) - 1

# Calculate CAGR for Gold and CPI 
gold_usd_cagr = calculate_cagr(df_monthly['Gold_USD'])
cpi_usa_cagr = calculate_cagr(df_monthly['CPI_USA'])
gold_inr_cagr = calculate_cagr(df_monthly['Gold_INR'])
cpi_india_cagr = calculate_cagr(df_monthly['CPI_India'])
gold_eur_cagr = calculate_cagr(df_monthly['Gold_EUR'])
cpi_germany_cagr = calculate_cagr(df_monthly['CPI_Germany'])

labels = ['USA', 'India', 'Germany']
gold = [gold_usd_cagr*100, gold_inr_cagr*100, gold_eur_cagr*100]
inf = [cpi_usa_cagr*100, cpi_india_cagr*100, cpi_germany_cagr*100]
x = np.arange(len(labels))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - 0.17, gold, 0.35, label='Gold CAGR %', alpha=0.35)
b2 = ax.bar(x + 0.17, inf, 0.35, label='Inflation CAGR %', alpha=0.35, color='darkblue')

# Add Labels & Real Gain Text
for i in range(len(labels)):
    ax.bar_label(b1, padding=3, fmt='%.2f%%', alpha=0.35)
    ax.bar_label(b2, padding=3, fmt='%.2f%%', alpha=0.35)
    ax.text(i, max(gold[i], inf[i]) + 1.5, f"Real Gain:\n+{gold[i]-inf[i]:.2f}%", 
            ha='center')

ax.set_title('Gold vs Inflation (2000-2026)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Annualized Rate (%)')
ax.legend()
plt.ylim(0, max(gold) + 4)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_path = os.path.join(charts_dir, 'Gold_vs_Inflation(2000-2026).png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 3. Correlation Analysis: Inter-Asset Relationships

This section uses a heatmap to quantify how Gold, Equity Indices, and Inflation (CPI) move in relation to one another across the three target regions.

### Key Details:
* **Asset Synergy:** Identifies whether Gold moves in tandem with local inflation or acts as a diversifier against stock market volatility.
* **Correlation Coefficient ($r$):** Uses a scale from -1.0 (opposite movement) to +1.0 (identical movement), with **0** representing no relationship.
* **Diversification Insight:** Helps determine if Gold provides a "hedge" (low or negative correlation) during periods when equity markets (S&P 500, DAX, Nifty50) are underperforming.
* **Regional Nuance:** Compares how closely localized Gold (e.g., Gold_INR) tracks with local inflation (CPI_India) versus global prices.

In [ ]:
# 1. Core columns for the correlation study
corr_cols = ['Gold_USD', 'Gold_INR', 'Gold_EUR', 'CPI_USA', 'CPI_India', 'CPI_Germany', 'SP500', 'DAX', 'Nifty50']
correlation_matrix = df_returns[corr_cols].corr()

# 2. Plot the Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdYlGn', center=0)
plt.title('Correlation Matrix: Does Gold actually move with Inflation?')
save_path = os.path.join(charts_dir, 'Correlation_Matrix.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 4. Risk-Adjusted Performance: Sharpe Ratio vs. Volatility

This section calculates the **Sharpe Ratio** to determine which assets provided the best return relative to the risk taken, while accounting for regional **Risk-Free Rates ($R_f$)**.

### Key Details:
* **Risk-Free Rate Differentiation:** Uses localized bench i.e 10 year bond yieldmarks (USD: 3.24%, INR: 7.42%, EUR: 1.81%) to ensure the "Excess Return" is calculated accurately for each specific economy.
* **Annualized Volatility:** Measures "Market Turbulence" by standardizing the standard deviation of monthly returns into an annual percentage.
* **The Sharpe Formula:** Calculated as $\frac{\text{Annualized Return} - R_f}{\text{Volatility}}$. A higher Sharpe ratio indicates superior investment efficiency.
* **Dual-Axis Visualization:** Combines a bar chart (Efficiency/Sharpe) with a line graph (Risk/Volatility) to visually identify assets that offer "High Reward for Low Stress."

In [ ]:
# 1. Configuration
full_metrics_cols = ['Gold_USD', 'Gold_INR', 'Gold_EUR', 'SP500', 'Nifty50', 'DAX']
rf_rates = {'USD': 3.24, 'INR': 7.42, 'EUR': 1.81}

# 2. Calculate Annualized Volatility (Standard Deviation)
volatility = df_returns[full_metrics_cols].std() * np.sqrt(12) * 100
sharpe_values = {}

# 3. Optimized Loop for Sharpe Calculation
for col in full_metrics_cols:
    ann_ret = df_returns[col].mean() * 12 * 100
    
    if 'INR' in col or 'Nifty50' in col:
        rf = rf_rates['INR']
    elif 'EUR' in col or 'DAX' in col:
        rf = rf_rates['EUR']
    else:
        rf = rf_rates['USD']
        
    sharpe_values[col] = (ann_ret - rf) / volatility[col]

metrics_df = pd.DataFrame({
    'Vol': volatility, 
    'Sharpe': pd.Series(sharpe_values)
}).sort_values('Sharpe', ascending=False)

fig, ax1 = plt.subplots(figsize=(8, 6))

# Primary Axis: Sharpe
bars = ax1.bar(metrics_df.index, metrics_df['Sharpe'], alpha=0.3, label='Sharpe Ratio')
ax1.set_ylabel('Sharpe Ratio (Efficiency)', fontsize=10)
ax1.bar_label(bars, fmt='%.2f', padding=3)

# Secondary Axis: Volatility 
ax2 = ax1.twinx()
ax2.plot(metrics_df.index, metrics_df['Vol'], color='red', marker='o', markersize=8, linewidth=2, label='Volatility %')
ax2.set_ylabel('Annualized Volatility (Risk)', fontsize=10)

# Formatting
ax1.set_title('Asset Efficiency vs. Market Turbulence (2000-2026)', fontsize=11, fontweight='bold', pad=15)
ax1.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
save_path = os.path.join(charts_dir, 'Efficiency_vs_Volatility.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 5. Dynamic Efficiency: Rolling 3-Year Sharpe Ratio

This section analyzes how asset efficiency changes over time using a **rolling window** approach, moving beyond static averages to reveal performance consistency.

### Key Details:
* **Rolling Window Logic:** Uses a 36-month (3-year) sliding window to calculate the Sharpe Ratio at every point in the 26-year timeline. This highlights periods of overperformance and underperformance.
* **Dynamic Risk-Free Rates:** The loop applies region-specific $R_f$ rates (INR, EUR, or USD) to each asset to maintain localized accuracy throughout the time series.
* **The "Zero Line" Benchmark:** Includes a horizontal reference line at 0.0. Points above this line indicate the asset is outperforming "safe" cash/bonds, while points below indicate a failure to compensate for market risk.
* **Consistency Check:** This visualization is critical for identifying whether an asset’s high total return was a result of a single "lucky" spike or a sustained period of superior risk-adjusted growth.

In [ ]:
# --- MANUAL ASSET TOGGLE ---
# Just add or remove names in this list to update the chart
my_selection = ['Gold_USD', 'Gold_INR', 'Gold_EUR'] ##['Gold_USD', 'Gold_INR', 'Gold_EUR', 'SP500', 'Nifty50', 'DAX']

window = 36
plt.figure(figsize=(12, 6))

for col in my_selection:
    rf = rf_rates['INR']/100 if 'INR' in col or 'Nifty' in col else \
         rf_rates['EUR']/100 if 'EUR' in col or 'DAX' in col else rf_rates['USD']/100
    
    rol_ret = df_returns[col].rolling(window=window).mean() * 12
    rol_vol = df_returns[col].rolling(window=window).std() * np.sqrt(12)
    rol_sharpe = (rol_ret - rf) / rol_vol
    
    plt.plot(rol_sharpe, label=col, linewidth=2)

plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title(f'Rolling 3Y Sharpe Ratio Comparison', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.2)
save_path = os.path.join(charts_dir, 'RollingSharpe_Sp500_Nifty50_Dax.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 6. Comparative Efficiency: The Modern Portfolio View

This section provides a side-by-side comparison of asset performance, distilling the complex relationship between risk and reward into two distinct visualizations.

### Key Details:
* **Efficiency Ranking (Bar Chart):** Ranks assets by their **Sharpe Ratio**, This provides an immediate "leaderboard" showing which assets generated the most excess return per unit of risk.
* **Risk vs. Return (Scatter Plot):** This chart identifies the "Efficient Frontier." Assets located toward the **top-left** are superior (higher returns with lower risk), while those toward the **bottom-right** are less desirable.
* **Cross-Asset Context:** Allows for a direct comparison between localized Gold (INR/EUR) and major equity benchmarks (Nifty50/DAX) to see which asset class truly dominates the risk-reward landscape.

In [ ]:
risk_metrics = metrics_df.rename(columns={'Vol': 'Volatility (%)', 'Sharpe': 'Sharpe Ratio'})
risk_metrics_sorted = risk_metrics.sort_values(by='Sharpe Ratio', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Sharpe Ratio (Efficiency) ---
colors = plt.cm.RdYlGn(np.linspace(0.8, 0.15, len(risk_metrics_sorted)))
risk_metrics_sorted['Sharpe Ratio'].plot(kind='bar', ax=ax1, color=colors)
ax1.set_title('Asset Efficiency: Sharpe Ratio', fontsize=12, fontweight='bold')
ax1.set_ylabel('Sharpe Ratio')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# --- Plot 2: Risk vs. Return  ---
for asset in risk_metrics.index:
    ann_return = df_returns[asset].mean() * 12 * 100
    ann_vol = risk_metrics.loc[asset, 'Volatility (%)']
    
    ax2.scatter(ann_vol, ann_return, s=150, label=asset, alpha=0.3)
    ax2.annotate(asset, (ann_vol + 0.15, ann_return), fontsize=10)

# Reference labels to explain the chart
ax2.set_title('Risk vs. Return', fontsize=12, fontweight='bold')
ax2.set_xlabel('Volatility (Annual Risk %)')
ax2.set_ylabel('Annualized Return %')
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
save_path = os.path.join(charts_dir, 'Efficiency_Risk_Vs_Return.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 7. Gold Market Liquidity: Trading Volume Trends

This section examines the trading volume of Gold from 2000 to 2026 to assess market liquidity and identify periods of heightened investor activity.

### Key Details:
* **Monthly Aggregation:** Uses the sum of trading volume per month to smooth out daily noise while capturing significant shifts in market interest.
* **12-Month Moving Average (MA):** Provides a trend line that filters out seasonal fluctuations, revealing the long-term momentum of Gold market participation.
* **Liquidity & Crisis Detection:** Significant spikes in volume often correlate with global financial uncertainty or "flights to safety," providing context for the volatility seen in previous sections.
* **Validation of Price Moves:** High volume during price rallies suggests strong institutional conviction, whereas low-volume rallies may indicate weaker market trends.

In [ ]:
gold_vol_monthly = df['Gold_Volume'] .resample('ME').sum()
vol_ma = gold_vol_monthly.rolling(window=12).mean()

# Plotting
plt.figure(figsize=(14, 6))

# Bar chart for monthly volume
plt.bar(gold_vol_monthly.index, gold_vol_monthly, alpha=0.5, 
        width=20, label='Monthly Trading Volume')

# Line chart for the 1-year trend
plt.plot(vol_ma.index, vol_ma, linewidth=2, 
         label='12-Month Moving Average')

# Formatting
plt.title('Gold Market Liquidity (2000-2026): Trading Volume', fontsize=14, fontweight='bold')
plt.ylabel('Volume (Total Monthly Shares/Ounces)')
plt.xlabel('Year')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend()
plt.tight_layout()
save_path = os.path.join(charts_dir, 'Volume.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 8. Global Return Attribution: Deconstructing the Gains

This section performs a performance attribution analysis to identify the specific drivers of Gold’s total return in different geographic regions.

### Key Details:
* **Factor Decomposition:** Breaks down total local profit into three distinct mathematical buckets:
    1. **Global Gold Price (USD):** The baseline appreciation of the asset on the world market.
    2. **Currency Devaluation:** The additional gain (or loss) caused by the local currency's movement against the US Dollar.
    3. **Compounding Bonus (Interaction):** The exponential growth caused by holding an appreciating asset in a strengthening currency.
* **The "Currency Hedge" Proof:** Visually demonstrates how investorsd Germany benefit from Gold not just as a commodity, but as a protective shield against local currency depreciation.
* **Cross-Regional Logic:** Compares a single-currency environment (USA) against multi-currency environments to show why Gold is a more powerful wealth preservation tool in emerging or fluctuatinld.g markets.

In [ ]:
def get_attribution(gold_col, curr_col, region_name):
    # Total Multipliers
    total_mult = df[gold_col].iloc[-1] / df[gold_col].iloc[0]
    gold_usd_mult = df['Gold_USD'].iloc[-1] / df['Gold_USD'].iloc[0]
    
    # If US, currency multiplier is 1.0
    if curr_col == 'USD_USD':
        curr_mult = 1.0
    else:
        curr_mult = df[curr_col].iloc[-1] / df[curr_col].iloc[0]
    
    pure_gold_gain = (gold_usd_mult - 1) * 100
    currency_gain = (curr_mult - 1) * 100
    total_gain = (total_mult - 1) * 100
    interaction = total_gain - (pure_gold_gain + currency_gain)
    
    return [region_name, pure_gold_gain, currency_gain, interaction, total_gain]

# Create the comparison table
data = [
    get_attribution('Gold_USD', 'USD_USD', 'USA (USD)'),
    get_attribution('Gold_INR', 'USD_INR', 'India (INR)'),
    get_attribution('Gold_EUR', 'USD_EUR', 'Germany (EUR)')
]

attribution_df = pd.DataFrame(data, columns=['Region', 'Gold_USD_Gain%', 'Currency_Gain%', 'Compounding_Bonus%', 'Total_Local_Gain%'])

plot_df = attribution_df.set_index('Region')
cols_to_plot = ['Gold_USD_Gain%', 'Currency_Gain%', 'Compounding_Bonus%']

# Stacked Bar Chart
ax = plot_df[cols_to_plot].plot(
    kind='bar', 
    stacked=True, 
    figsize=(8, 6), 
    alpha=0.55
)

plt.title('Why Your Gold Portfolio Grew: Return Attribution (2000-2026)', fontsize=10, fontweight='bold', pad=15)
plt.ylabel('Total Percentage Gain (%)', fontsize=10)
plt.xlabel('Investment Region', fontsize=10)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Add "Total Gain" labels on top of bars
for i, total in enumerate(plot_df['Total_Local_Gain%']):
    plt.text(i, total + 10, f'{total:,.1f}%', ha='center')

plt.legend(['Global Gold Price (USD)', 'Currency Devaluation', 'Compounding Bonus'], 
           loc='upper left', frameon=True)
plt.tight_layout()
save_path = os.path.join(charts_dir, 'Growth_IND_USA_DE.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 9. Global Purchasing Power: The Inflation Gap Analysis

This section compares the "Nominal" price (market sticker price) against the "Real" price (inflation-adjusted) for the USA, India, and Germany.

### Key Details:
* **The "Shaded Gap":** Represents the cumulative loss of currency value. A wider gap indicates higher local inflation and a greater need for Gold as a hedge.
* **Regional Comparison:** * **India:** Typically shows the widest gap, highlighting the Rupee's historical struggle against inflation.
    * **USA/Germany:** Show narrower gaps, indicating higher currency stability relative to the Gold standard.
* **The Investor's Reality:** If the "Real Gold" line is flat while the "Nominal" line is rising, the investor is not gaining wealth—they are simply not losing it.

In [ ]:
regions = [
    ('USA', 'Gold_USD', 'Real_Gold_USD', 'USD', '#2ecc71'), # Green
    ('India', 'Gold_INR', 'Real_Gold_INR', 'INR', '#f39c12'), # Orange
    ('Germany', 'Gold_EUR', 'Real_Gold_EUR', 'EUR', '#3498db') # Blue
]

fig, axes = plt.subplots(3, 1, figsize=(12, 18), sharex=True)

for i, (name, nom_col, real_col, curr, color) in enumerate(regions):
    ax = axes[i]
    
    # Plot Nominal vs Real
    ax.plot(df_monthly.index, df_monthly[nom_col], label=f'Nominal Gold ({curr})', color=color, alpha=0.4, linestyle='--')
    ax.plot(df_monthly.index, df_monthly[real_col], label=f'Real Gold (Adj. for CPI)', color=color, linewidth=2.5)
    
    # Fill the gap
    ax.fill_between(df_monthly.index, df_monthly[nom_col], df_monthly[real_col], color='gray', alpha=0.15)
    
    ax.set_title(f'Gold Purchasing Power: {name}', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'Price in {curr}')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.2)

plt.xlabel('Year')
plt.tight_layout()
save_path = os.path.join(charts_dir, 'Nominal_vs_Real_Global_Comparison.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()

## 10. Realistic Wealth Accumulation: Dollar Cost Averaging (DCA) with Friction

This section simulates a monthly investment strategy (DCA) of **€100**, comparing a direct investment in Germany against a cross-border investment in India while accounting for real-world transaction costs.

### Key Details:
* **Friction and "Drift":** Unlike previous theoretical models, this simulation applies **Investment-Grade Parameters** such as India's customs duties, GST (3%), and bank premiums (totaling ~10% friction), alongside currency transfer spreads (0.5%).
* **Cross-Border Simulation:**
    1. **Germany Strategy:** Direct monthly purchase of Gold at the international EUR spot price.
    2. **India Strategy:** Converts EUR to INR, pays the transfer fee, and purchases Gold at the localized (higher) Indian price.
* **The "Efficiency Loss" Verdict:** By converting the final Indian portfolio value back into EUR, the model calculates the **Total Efficiency Loss**. This represents the "cost of geography"—how much wealth is eroded by local taxes and transfer fees over 26 years.
* **Wealth Accumulation Plot:** Visualizes the "drift" between the two portfolios. While both follow the same price trend, the gap between the solid blue line (Germany) and the dashed red line (India) quantifies the cumulative impact of investment friction.

In [ ]:
# 1. Define Realistic Investment-Grade Parameters
# 6% Basic Customs Duty + 3% GST + ~1% Bank/Mint Premium = ~10% Total Friction
india_investment_friction = 0.010  
transfer_spread = 0.005  # 0.5% fee/spread for EUR -> INR conversion

monthly_euro_investment = 100
germany_portfolio_eur = []
india_portfolio_inr = []
total_ounces_ger = 0
total_ounces_ind = 0

# 2. DCA Simulation Loop
for date, row in df_monthly.iterrows():
    # --- Germany Scenario ---
    # Buy gold at international EUR spot
    total_ounces_ger += monthly_euro_investment / row['Gold_EUR']
    germany_portfolio_eur.append(total_ounces_ger * row['Gold_EUR'])
    
    # --- India Scenario ---
    # Step A: Convert EUR to INR
    eur_inr_rate = row['USD_INR'] / row['USD_EUR']
    inr_to_invest = (monthly_euro_investment * (1 - transfer_spread)) * eur_inr_rate
    
    # Step B: Buy Gold in India
    local_india_price = row['Gold_INR'] * (1 + india_investment_friction)
    total_ounces_ind += inr_to_invest / local_india_price
    
    # Store nominal value in INR
    india_portfolio_inr.append(total_ounces_ind * row['Gold_INR'])

# 3. Save results to df_monthly
df_monthly['Wealth_Germany_EUR'] = germany_portfolio_eur
df_monthly['Wealth_India_INR'] = india_portfolio_inr
df_monthly['Wealth_India_converted_to_EUR'] = df_monthly['Wealth_India_INR'] / (df_monthly['USD_INR'] / df_monthly['USD_EUR'])

# 4. Final Comparison
final_ger = df_monthly['Wealth_Germany_EUR'].iloc[-1]
final_ind = df_monthly['Wealth_India_converted_to_EUR'].iloc[-1]
percentage_drag = (1 - (final_ind / final_ger)) * 100

print(f"--- Updated Investment-Grade Verdict ---")
print(f"Final Wealth Germany Strategy: €{final_ger:,.2f}")
print(f"Final Wealth India Strategy:    €{final_ind:,.2f}")
print(f"Total Efficiency Loss: {percentage_drag:.2f}%")

# 5. Plotting the "Drift"
plt.figure(figsize=(12, 6))
plt.plot(df_monthly.index, df_monthly['Wealth_Germany_EUR'], label='Germany (Pure Market Price)', color='#2980b9', linewidth=2)
plt.plot(df_monthly.index, df_monthly['Wealth_India_converted_to_EUR'], label='India (Reflecting GST & Duties)', color='#c0392b', linestyle='--')
plt.title('Wealth Accumulation: The Impact of Friction on Investment Grade Gold')
plt.ylabel('Value in EUR')
plt.legend()
plt.grid(True, alpha=0.2)
save_path = os.path.join(charts_dir, 'Wealth_Accumulation_DCA.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()